# P12 Statistical Rigor Reproducible

由原 P12 拆出：整理五個 seeds 的信賴區間、校正後 p 值與成對效果量。


## 0. Setup


In [1]:
from __future__ import annotations

from pathlib import Path
from statistics import mean, median, stdev
from collections import Counter, defaultdict
from datetime import datetime
import csv
import json
import math
import re

try:
    from scipy import stats as scipy_stats
    SCIPY_AVAILABLE = True
except Exception:
    scipy_stats = None
    SCIPY_AVAILABLE = False

BASE = Path("/home/ester/valid_description")
TGT = BASE / "experiment data" / "journal_figures_tables_20260629"
FIG = TGT / "figures"
TAB = TGT / "tables"
DER = TGT / "derived_audits"
SRC = TGT / "source_programs"
REPO = BASE / "text-conditioned-outfit-recommendation"
EXP_MAIN = REPO / "experiments"
EXP_NONE = REPO / "experiments_none_only"

for p in [FIG, TAB, DER, SRC]:
    p.mkdir(parents=True, exist_ok=True)

TOKEN_RE = re.compile(r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)?")
WORD_RE = re.compile(r"[A-Za-z]+(?:[-'][A-Za-z]+)?")

def read_json(path: Path):
    with path.open(encoding="utf-8") as f:
        return json.load(f)

def read_csv(path: Path) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))

def write_csv(path: Path, rows: list[dict]) -> None:
    if not rows:
        raise ValueError(f"No rows to write: {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print("[write]", path)

def display_csv(path: Path, max_rows: int = 12) -> None:
    rows = read_csv(path)
    for r in rows[:max_rows]:
        print(r)
    if len(rows) > max_rows:
        print(f"... ({len(rows) - max_rows} more rows)")

def fmt_float(x, nd=4) -> str:
    try:
        if x is None or math.isnan(float(x)):
            return ""
        return f"{float(x):.{nd}f}"
    except Exception:
        return ""

def fmt_pct(num, den) -> str:
    return f"{100 * float(num) / float(den):.2f}" if den else "0.00"

def to_int(x, default=0) -> int:
    try:
        return int(float(x))
    except Exception:
        return default

def to_float(x, default=0.0) -> float:
    try:
        return float(x)
    except Exception:
        return default

def query_token_count(text: str) -> int:
    return len(TOKEN_RE.findall(str(text or "").strip()))

def word_tokens(text: str) -> list[str]:
    return [t.lower() for t in WORD_RE.findall(str(text or "").strip())]

def word_count(text: str) -> int:
    return len(word_tokens(text))

def original_field(original_data: dict, set_id: str, field: str) -> str:
    value = original_data.get(str(set_id), "")
    if isinstance(value, dict):
        return str(value.get(field, "") or "").strip()
    return str(value or "").strip()

def original_url_name_plus_title(original_data: dict, set_id: str) -> str:
    value = original_data.get(str(set_id), "")
    if isinstance(value, dict):
        parts = [str(value.get("url_name", "") or "").strip(), str(value.get("title", "") or "").strip()]
        return " ".join(p for p in parts if p).strip()
    return str(value or "").strip()

def generated_title_only(generated_data: dict, set_id: str) -> str:
    value = generated_data.get(str(set_id), "")
    if isinstance(value, dict):
        return str(value.get("title", "") or "").strip()
    return str(value or "").strip()

print("BASE =", BASE)
print("TAB  =", TAB)
print("DER  =", DER)
print("SCIPY_AVAILABLE =", SCIPY_AVAILABLE)


BASE = /home/ester/valid_description
TAB  = /home/ester/valid_description/experiment data/journal_figures_tables_20260629/tables
DER  = /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits
SCIPY_AVAILABLE = True


## 5. Statistical Stability Across Seeds


In [6]:
def dedup_exact(rows):
    seen, out = set(), []
    for r in rows:
        key = tuple(sorted(r.items()))
        if key not in seen:
            seen.add(key)
            out.append(r)
    return out

def by_seed(rows, model, feat, metric):
    return {int(r["seed"]): float(r[metric]) for r in rows if r.get("model_tag") == model and r.get("feat_tag") == feat}

def paired_row(task, metric, baseline, proposed):
    seeds = sorted(set(baseline) & set(proposed))
    b = [baseline[s] for s in seeds]
    p = [proposed[s] for s in seeds]
    diffs = [pp - bb for bb, pp in zip(b, p)]
    n = len(diffs)
    dmean = mean(diffs)
    dsd = stdev(diffs) if n >= 2 else 0.0
    dz = dmean / dsd if dsd else float("nan")
    if SCIPY_AVAILABLE and n >= 2:
        pt = float(scipy_stats.ttest_rel(p, b).pvalue)
        try:
            pw = float(scipy_stats.wilcoxon(diffs, zero_method="wilcox").pvalue)
        except Exception:
            pw = float("nan")
        tcrit = float(scipy_stats.t.ppf(0.975, n - 1))
    else:
        pt = pw = tcrit = float("nan")
    se = dsd / (n ** 0.5) if n and dsd else 0.0
    return {
        "comparison": "Full contextual rewrite vs original description",
        "task": task,
        "metric": metric,
        "baseline_mean": fmt_float(mean(b), 6),
        "baseline_std": fmt_float(stdev(b), 6) if len(b) >= 2 else "",
        "proposed_mean": fmt_float(mean(p), 6),
        "proposed_std": fmt_float(stdev(p), 6) if len(p) >= 2 else "",
        "mean_difference": fmt_float(dmean, 6),
        "ci95_low": fmt_float(dmean - tcrit * se, 6),
        "ci95_high": fmt_float(dmean + tcrit * se, 6),
        "paired_t_p": fmt_float(pt, 6),
        "wilcoxon_p": fmt_float(pw, 6),
        "cohens_dz": fmt_float(dz, 6),
        "paired_seeds": ",".join(str(s) for s in seeds),
        "n_seeds": str(n),
        "source_scope": "main experiments/ sweep results after exact duplicate removal",
    }

def bh(rows, pcol, outcol):
    vals = [(i, to_float(r[pcol], float("nan"))) for i, r in enumerate(rows)]
    vals = [(i, p) for i, p in vals if not math.isnan(p)]
    m = len(vals)
    for r in rows:
        r[outcol] = ""
    prev = 1.0
    for rank_from_end, (i, p) in enumerate(reversed(sorted(vals, key=lambda x: x[1])), start=1):
        rank = m - rank_from_end + 1
        adj = min(prev, p * m / rank)
        rows[i][outcol] = fmt_float(adj, 6)
        prev = adj

cp_raw = read_csv(EXP_MAIN / "results_cp_sweep.csv")
cir_raw = read_csv(EXP_MAIN / "results_cir_sweep.csv")
cp = dedup_exact(cp_raw)
cir = dedup_exact(cir_raw)
a14 = []
for metric in ["auc", "fitb_acc"]:
    a14.append(paired_row("CP", metric, by_seed(cp, "cp_old", "old", metric), by_seed(cp, "cp_new", "new", metric)))
for metric in ["recall_at_10", "recall_at_30", "recall_at_50"]:
    a14.append(paired_row("CIR", metric, by_seed(cir, "cir_old", "old", metric), by_seed(cir, "cir_new", "new", metric)))
bh(a14, "paired_t_p", "paired_t_p_bh")
bh(a14, "wilcoxon_p", "wilcoxon_p_bh")
for r in a14:
    r["raw_rows_before_dedup"] = str(len(cp_raw) if r["task"] == "CP" else len(cir_raw))
    r["rows_after_exact_dedup"] = str(len(cp) if r["task"] == "CP" else len(cir))
    r["dedup_note"] = "Seed 1 had duplicate appended sweep rows; exact duplicates were removed before statistical tests."
write_csv(DER / "A14_statistical_rigor_ci_adjusted_p_effect_sizes.csv", a14)
display_csv(DER / "A14_statistical_rigor_ci_adjusted_p_effect_sizes.csv")


[write] /home/ester/valid_description/experiment data/journal_figures_tables_20260629/derived_audits/A14_statistical_rigor_ci_adjusted_p_effect_sizes.csv
{'comparison': 'Full contextual rewrite vs original description', 'task': 'CP', 'metric': 'auc', 'baseline_mean': '0.929165', 'baseline_std': '0.001302', 'proposed_mean': '0.945370', 'proposed_std': '0.001251', 'mean_difference': '0.016205', 'ci95_low': '0.015688', 'ci95_high': '0.016722', 'paired_t_p': '0.000000', 'wilcoxon_p': '0.062500', 'cohens_dz': '38.934190', 'paired_seeds': '1,2,3,4,5', 'n_seeds': '5', 'source_scope': 'main experiments/ sweep results after exact duplicate removal', 'paired_t_p_bh': '0.000000', 'wilcoxon_p_bh': '0.062500', 'raw_rows_before_dedup': '24', 'rows_after_exact_dedup': '20', 'dedup_note': 'Seed 1 had duplicate appended sweep rows; exact duplicates were removed before statistical tests.'}
{'comparison': 'Full contextual rewrite vs original description', 'task': 'CP', 'metric': 'fitb_acc', 'baseline_mea